# Import Statements

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import yfinance as yf

from modules.utils import *
from modules.screen import *
from modules.data_loader import *
from modules.portfolio import *
from modules.trade import *
from modules.backtest import *
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from modules.tearsheet import Tearsheet

import plotly.io as pio
pio.renderers.default = "notebook_connected"

---
# Parameters

In [ ]:
config = load_config("config.yaml")

# Extract config parameters
start_date = config["start_date"]
end_date = config["end_date"]
BENCHMARK = config["benchmark"]
UPDATE = config["update"]
SCREEN = config["screen"]

---
# Data Fetching

In [ ]:
# Loading data for all NSE tickers and calculating features
full_nse_tickers = load_nse_all()
load_full_data = load_data(start_date=start_date, end_date=end_date, update=UPDATE, full_nse_tickers=full_nse_tickers,
                           benchmark=BENCHMARK)
# load benchmark data
benchmark_prices = download_benchmark_data(symbol=BENCHMARK)
priceData = (
    load_full_data
    .reset_index()[['Date', 'TIC', 'Adj Close']]
    .set_index(['Date', 'TIC'])
    .unstack(level=1)
    .droplevel(axis=1, level=0)
    .ffill()
    .reset_index()
)

# revert back to original format for backtest
priceData = (
    priceData
    .set_index("Date")
    .stack()
    .reset_index()
    .rename(columns={0: "Adj Close", "level_1": "TIC"})
)

---
# Backtest - Simulation

In [ ]:
backtest_df, trade_blotter, portfolio_history = run_backtest(
    price_df=priceData,
    screen_rule=SCREEN,
    initial_cash=config["cash"],
    rebalance_freq=config["rebalance_frequency"],
    feature_df=load_full_data,
    start_date=start_date,    
    end_date=end_date,
    allocator="max_sharpe",
    ranker_dict=config["ranker"]
)

---
# Portfolio Analysis

In [ ]:
pv = backtest_df.copy()
pv["Date"] = pd.to_datetime(pv["Date"])
pv = pv.set_index("Date")["equity"]

bv = benchmark_prices.copy()
bv["Date"] = pd.to_datetime(bv["Date"])
bv = bv.set_index("Date")["bench"]

# align with portfolio dates
bv = bv.reindex(pv.index).ffill()

In [ ]:
ts = Tearsheet(
    portfolio_values = pv,
    benchmark_values = bv,
    trades = None,   # ignore trades for now,
    risk_free_rate = 0.06
)

# All stats as Plotly tables (show inline in Jupyter/VSCode)
tables = ts.summary()
display(tables["summary"])

# Everything at once
ts.plot_all()